In [1]:
import pandas as pd
import numpy as np
import zipfile

zip_path = "verdicts_alina.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    with z.open("verdicts_alina.csv") as f:
        df = pd.read_csv(f)

print(df.shape)

(34765, 23)


In [2]:
import re

def count_killers(names_text: str) -> int:
    text = names_text.replace('\n', ' ').replace('\r', ' ').strip()

    people = re.split(r'\s*,\s*(?=[^,]*?\s*-\s*ст\.)', text)

    count = 0
    for person in people:
        # print(person)
        if '105' in person:
            count += 1
    # print("end")
    return count

In [3]:
df['killer_count'] = df['names'].apply(count_killers)
df['killer_count'].value_counts()

killer_count
1    34765
Name: count, dtype: int64

In [4]:
import re

def estimate_victims(names_text: str) -> int:
    if not isinstance(names_text, str):
        return 0

    pattern = r"ст\. ?\d+(?: ч\.\d+)?(?: п\.[п.]?[\w.,]+)?"
    articles = re.findall(pattern, names_text)

    flag = False

    for article in articles:
        cleaned = re.sub(r"[.\s]", "", article).lower()

        if "105" in cleaned:
            if "ч2" in cleaned:
                if "па" in cleaned:
                    flag = True
    if flag:
        return 1
    else:
        return 0


text1 = "Петров И.И. - ст.105 ч.1 УК РФ"
text2 = "Иванов С.С. - ст.105 ч.2 п.а, п.ж УК РФ"
text3 = "Сидоров - ст. 105 ч.2 п.п.а УК РФ"

print(estimate_victims(text1)) 
print(estimate_victims(text2))
print(estimate_victims(text3))

0
1
1


In [5]:
df['many_victims'] = df['names'].apply(estimate_victims)
df['many_victims'].value_counts()

many_victims
0    32887
1     1878
Name: count, dtype: int64

In [6]:
df["entryDate"] = pd.to_datetime(df["entryDate"])
df["year"] = df["entryDate"].dt.year

def get_season(month):
    if month in [12, 1, 2]:
        return "Зима"
    elif month in [3, 4, 5]:
        return "Весна"
    elif month in [6, 7, 8]:
        return "Лето"
    else:
        return "Осень"

df["season"] = df["entryDate"].dt.month.apply(get_season)

In [7]:
df.columns

Index(['id', 'region', 'entryDate', 'names', 'judge', 'decision', 'accused',
       'articles', 'link_text', 'preamble', 'description', 'sentence',
       'killer_count', 'gender_accused', 'prior_convictions', 'alcohol',
       'precrime_argument', 'has_woman_victim', 'has_man_victim', 'method',
       'motive', 'location', 'prison_term', 'many_victims', 'year', 'season'],
      dtype='object')

In [8]:
df = df.drop(['preamble', 'description', 'sentence', 'link_text', 'accused', 'articles', 'entryDate'], axis=1)

In [9]:
df['decision'].value_counts()

decision
Вынесен ПРИГОВОР                                        34611
Вступило в силу                                           152
Применены ПРИНУДИТЕЛЬНЫЕ МЕРЫ МЕДИЦИНСКОГО ХАРАКТЕРА        2
Name: count, dtype: int64

In [10]:
df.columns

Index(['id', 'region', 'names', 'judge', 'decision', 'killer_count',
       'gender_accused', 'prior_convictions', 'alcohol', 'precrime_argument',
       'has_woman_victim', 'has_man_victim', 'method', 'motive', 'location',
       'prison_term', 'many_victims', 'year', 'season'],
      dtype='object')

In [11]:
import re

def estimate_attempt(names_text: str) -> int:
    if not isinstance(names_text, str):
        return 0

    pattern = r"ст\. ?\d+(?: ч\.\d+)?(?: п\.[п.]?[\w.,]+)?"
    articles = re.findall(pattern, names_text)

    flag = False

    for article in articles:
        cleaned = re.sub(r"[.\s]", "", article).lower()
        #print(cleaned)
        if "30" in cleaned:
            flag = True
    if flag:
        return 1
    else:
        return 0

text1 = "Петров И.И. - ст.30 ч.1 УК РФ"
text2 = "Иванов С.С. - ст.105 ч.2 п.а, п.ж УК РФ"
text3 = "Сидоров - ст. 105 ч.2 п.п.а УК РФ"

print(estimate_attempt(text1)) 
print(estimate_attempt(text2))
print(estimate_attempt(text3))

1
0
0


In [12]:
df['attempt'] = df['names'].apply(estimate_attempt)
df['attempt'].value_counts()

attempt
0    26917
1     7848
Name: count, dtype: int64

In [13]:
import re

def estimate_victim_child_or_helpless(names_text: str) -> int:
    if not isinstance(names_text, str):
        return 0

    pattern = r"ст\. ?\d+(?: ч\.\d+)?(?: п\.[п.]?[\w.,]+)?"
    articles = re.findall(pattern, names_text)

    flag = False

    for article in articles:
        cleaned = re.sub(r"[.\s]", "", article).lower()
        #print(cleaned)

        if "105" in cleaned:
            if "ч2" in cleaned:
                if "в" in cleaned:
                    flag = True
    if flag:
        return 1
    else:
        return 0

df['victim_child_or_helpless'] = df['names'].apply(estimate_victim_child_or_helpless)
df['victim_child_or_helpless'].value_counts()

victim_child_or_helpless
0    34187
1      578
Name: count, dtype: int64

In [14]:
import re

def estimate_cruelty(names_text: str) -> int:
    if not isinstance(names_text, str):
        return 0

    pattern = r"ст\. ?\d+(?: ч\.\d+)?(?: п\.[п.]?[\w.,]+)?"
    articles = re.findall(pattern, names_text)

    flag = False

    for article in articles:
        cleaned = re.sub(r"[.\s]", "", article).lower()
        #print(cleaned)

        if "105" in cleaned:
            if "ч2" in cleaned:
                if "д" in cleaned:
                    flag = True
    if flag:
        return 1
    else:
        return 0

df['cruel'] = df['names'].apply(estimate_cruelty)
df['cruel'].value_counts()

cruel
0    34115
1      650
Name: count, dtype: int64

In [15]:
df['region'].value_counts()

region
50    1674
2     1408
66    1388
24    1226
3     1138
      ... 
1       59
95      37
87      33
83      20
20       1
Name: count, Length: 85, dtype: int64

In [16]:
df['region'].min(), df['region'].max()

(1, 95)

In [17]:
region_map = {
    1: "Республика Адыгея",
    4: "Республика Алтай",
    2: "Республика Башкортостан",
    3: "Республика Бурятия",
    5: "Республика Дагестан",

    6: "Республика Ингушетия",
    7: "Кабардино-Балкарская Республика",
    8: "Республика Калмыкия",
    9: "Карачаево-Черкесская Республика",
    10: "Республика Карелия",

    11: "Республика Коми",
    82: "Республика Крым",
    12: "Республика Марий Эл",
    13: "Республика Мордовия",
    14: "Республика Саха (Якутия)",

    15: "Республика Северная Осетия — Алания",
    16: "Республика Татарстан",
    17: "Республика Тыва",
    18: "Удмуртская Республика",
    19: "Республика Хакасия",

    20: "Чеченская Республика",
    95: "Чеченская Республика",
    21: "Чувашская Республика",
    22: "Алтайский край",
    75: "Забайкальский край",
    41: "Камчатский край",

    23: "Краснодарский край",
    24: "Красноярский край",
    59: "Пермский край",
    25: "Приморский край",
    26: "Ставропольский край",

    27: "Хабаровский край",
    28: "Амурская область",
    29: "Архангельская область",
    30: "Астраханская область",
    31: "Белгородская область",

    32: "Брянская область",
    33: "Владимирская область",
    34: "Волгоградская область",
    35: "Вологодская область",
    36: "Воронежская область",

    37: "Ивановская область",
    38: "Иркутская область",
    39: "Калининградская область",
    40: "Калужская область",
    42: "Кемеровская область",

    43: "Кировская область",
    44: "Костромская область",
    45: "Курганская область",
    46: "Курская область",
    47: "Ленинградская область",

    48: "Липецкая область",
    49: "Магаданская область",
    50: "Московская область",
    51: "Мурманская область",
    52: "Нижегородская область",

    53: "Новгородская область",
    54: "Новосибирская область",
    55: "Омская область",
    56: "Оренбургская область",
    57: "Орловская область",

    58: "Пензенская область",
    60: "Псковская область",
    61: "Ростовская область",
    62: "Рязанская область",
    63: "Самарская область",

    64: "Саратовская область",
    65: "Сахалинская область",
    66: "Свердловская область",
    67: "Смоленская область",
    68: "Тамбовская область",

    69: "Тверская область",
    70: "Томская область",
    71: "Тульская область",
    72: "Тюменская область",
    73: "Ульяновская область",

    74: "Челябинская область",
    76: "Ярославская область",
    77: "Москва",
    78: "Санкт-Петербург",
    92: "Севастополь",

    79: "Еврейская автономная область",
    83: "Ненецкий автономный округ",
    80: "Донецкая народная республика",
    81: "Луганская народная республика",
    84: "Херсонская область",

    85: "Запорожская область",
    86: "Ханты-Мансийский автономный округ — Югра",
    87: "Чукотский автономный округ",
    89: "Ямало-Ненецкий автономный округ",
    94: "Байконур"
}

df["region_name"] = df["region"].map(region_map)
df["region_name"].value_counts()

region_name
Московская область                     1674
Республика Башкортостан                1408
Свердловская область                   1388
Красноярский край                      1226
Республика Бурятия                     1138
                                       ... 
Республика Северная Осетия — Алания      62
Республика Адыгея                        59
Чеченская Республика                     38
Чукотский автономный округ               33
Ненецкий автономный округ                20
Name: count, Length: 84, dtype: int64

In [18]:
for val, count in df["region_name"].value_counts().items():
    print(f"{val}, {count}")

Московская область, 1674
Республика Башкортостан, 1408
Свердловская область, 1388
Красноярский край, 1226
Республика Бурятия, 1138
Кемеровская область, 1078
Краснодарский край, 1069
Иркутская область, 999
Пермский край, 951
Челябинская область, 923
Забайкальский край, 920
Нижегородская область, 914
Алтайский край, 781
Ростовская область, 757
Хабаровский край, 732
Новосибирская область, 712
Архангельская область, 690
Приморский край, 595
Оренбургская область, 535
Санкт-Петербург, 510
Республика Саха (Якутия), 504
Ленинградская область, 500
Республика Татарстан, 492
Ставропольский край, 487
Омская область, 477
Саратовская область, 469
Тульская область, 430
Владимирская область, 406
Удмуртская Республика, 390
Курганская область, 386
Волгоградская область, 386
Республика Коми, 385
Тюменская область, 375
Республика Крым, 359
Вологодская область, 356
Ульяновская область, 344
Воронежская область, 334
Республика Тыва, 329
Кировская область, 316
Ивановская область, 301
Чувашская Республика, 290

In [21]:
df.columns

Index(['id', 'region', 'names', 'judge', 'decision', 'killer_count',
       'gender_accused', 'prior_convictions', 'alcohol', 'precrime_argument',
       'has_woman_victim', 'has_man_victim', 'method', 'motive', 'location',
       'prison_term', 'many_victims', 'year', 'season', 'attempt',
       'victim_child_or_helpless', 'cruel', 'region_name'],
      dtype='object')

In [22]:
df = df.drop(['names', 'killer_count', 'region'], axis=1)

In [23]:
df.columns

Index(['id', 'judge', 'decision', 'gender_accused', 'prior_convictions',
       'alcohol', 'precrime_argument', 'has_woman_victim', 'has_man_victim',
       'method', 'motive', 'location', 'prison_term', 'many_victims', 'year',
       'season', 'attempt', 'victim_child_or_helpless', 'cruel',
       'region_name'],
      dtype='object')

In [24]:
df.to_csv('df_labeled_all_final.csv', index=False)